# B2.3 · Vulnerability auditing: three generations of SAST

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.2 · Threat modelling from what the estate already knows](https://spbreed.github.io/cyber-commons/lessons/B2.2.html)**.

| | |
|---|---|
| Tools used | OpenGrep, Semgrep OSS, CodeQL, GLM-4.6, Kimi K2, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Score grep, taint rules and model review against the same corpus, then combine them behind a confidence gate.

**Why a security engineer needs it.** Pattern matching floods the queue; the false-positive rate is what actually changed. The control it builds is: stage 7: deterministic rules for what rules do well, model reasoning for what rules cannot express.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Three generations of static analysis are on the market and all three are sold with the same word. Pattern matching cannot follow a value; dataflow cannot read intent; a model can do both and will also tell you about a vulnerability that is not there.

> **At CyberTravels.** The IDOR that exposed card details by booking ID (R8) is exactly the class each generation of SAST handles differently — and the class the third generation will also confidently invent.

## 2 · The framework

```
   gen 1  pattern      grep-shaped     finds: the literal string
                                       misses: the same bug spelled differently

   gen 2  dataflow     source -> sink  finds: the value that reaches
                                       misses: intent, framework magic

   gen 3  reasoning    reads it        finds: both of the above
                                       adds:  confident findings that are not real

   the third generation does not replace the second. it needs it as an oracle.
```

**Stage 7 — Vulnerability auditing.** The deep-dive analysis stage, and the one
people think of as "SAST". It has had three generations, and knowing what each
can and cannot see is what stops you buying the wrong one.

**Generation 1 — grep.** Pattern-match dangerous constructs. Fast, zero setup,
fires on every occurrence whether reachable or not. Precision is poor, so it gets
muted.

**Generation 2 — rules with dataflow.** Semgrep, CodeQL, OpenGrep. Parse to an
AST or graph and track *taint*: does untrusted input reach a dangerous sink?
Precision improves enormously. The cost is that a rule only finds the pattern
someone wrote it for.

**Generation 3 — model review.** An open-weight model reads the code and reasons.
No rule needs to exist first, which is exactly its value — and it also invents
bugs that are not there, confidently.

The mistake is treating generation 3 as a replacement for generation 2. The
combination that works: rules for what rules do well, deterministically; the
model for what rules cannot express; and everything the model says treated as a
**hypothesis** until stages 8–12 confirm it.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Generation 1 — grep, and why it gets muted

The safe functions in this corpus matter more than the buggy ones: a scanner that fires on parameterised SQL is one nobody runs twice.

## 4 · Generation 2 — taint rules

The improvement is not a better pattern. It is a different question: *does untrusted input reach this sink?* A function parameter is untrusted; a string literal is not.

## 5 · Generation 3 — what rules structurally cannot see

Generation 2 is perfect on this corpus. So why involve a model? Because a rule only finds what someone wrote it for. Here is a bug with no rule: an authorization check that is *present* and wrong.

## 6 · Generation 2, as the tool you would actually run

The taint engine above is forty lines so it fits in a lesson. In production
generation 2 is Semgrep, CodeQL or OpenGrep, and a rule is a file. This is the
Semgrep rule for the same taint property the engine above implements:

```yaml
rules:
  - id: cybertravels-sql-concat
    languages: [python]
    severity: ERROR
    message: >-
      Traveller-controlled input is concatenated into a SQL string. Use a
      parameterised query.
    mode: taint
    pattern-sources:
      - pattern: $REQ.args[...]
      - pattern: $REQ.files[...]
    pattern-sinks:
      - pattern: $CONN.execute(...)
    pattern-sanitizers:
      - pattern: sqlite3.paramstyle
```

[`labs/tools/semgrep-sast/`](https://github.com/spbreed/cyber-commons/tree/main/labs/tools/semgrep-sast)
installs Semgrep 1.176.0 and runs it against a pull request from the Coding
Agent. Two things came out of that run and both matter here.

**Coverage is a configuration decision, and it is invisible.** The same file,
two ruleset widths:

```
  p/python + p/secrets: 1 finding
    line  17  ERROR   subprocess-shell-true

  seven packs: 4 findings
    line   9  ERROR   sqlalchemy-execute-raw-query
    line  14  WARNING eval-detected
    line  17  ERROR   subprocess-shell-true
    line  20  ERROR   disabled-cert-validation
```

Nothing about the file changed. On the narrow setting three real defects were
simply not looked for, and the scan exits 0 either way.

**And two defects survived both widths:**

```
  line  22  MISSED a live-looking API key on a module-level constant
  line   7  MISSED find_booking performs no authorisation check of any kind
```

The first is lexical — `p/secrets` was enabled and did not fire, because the
string matches no known provider's format. A rule could catch it, once someone
writes that rule. The second cannot be caught by any rule, because the defect is
the **absence** of a call in a function whose caller holds payments scope. That
is the boundary generation 3 exists to cross, and it is why the answer is
"both" rather than "the newer one".

## 7 · An agent drives both, because you cannot afford to run both everywhere

Generation 2 is cheap enough to run over the whole repository. Generation 3 is
not — at four million lines the model pass costs more than the finding is
worth, and a model asked to review everything reviews nothing carefully.

So neither generation is the interesting part. **The allocation is.** An agent
sits above both, and its policy is three rules:

1. run the deterministic scanner everywhere, with the widest ruleset that is
   not noisy, because it is nearly free;
2. spend the model pass only where stage 1 said risk lives **and** the rules
   were silent — silence in a high-risk zone is the signal, not the noise;
3. mark everything the model says as a hypothesis, never a finding, because
   stages 8 to 12 are what turn one into the other.

## 8 · The stage, as a skill

Three generations of analysis over the same CyberTravels code, and they fail differently: grep flags the safe queries, taint finds the real flows and nothing in `authz.py`, and the model finds the authorization defect that has no syntactic signature — along with the hallucination that is the price of it. The skill runs all three and reports precision, recall and that last column.

### The skill — [`skills/appsec/sast-generation-comparison/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/sast-generation-comparison/SKILL.md)

```yaml
name: sast-generation-comparison
description: >-
  Run pattern rules, taint analysis and a model over the same code and compare
  precision, recall and the class of defect only the third one finds. Use when
  choosing static analysis, justifying a model in the pipeline, or explaining
  why the scanner's output is mostly noise.
allowed-tools: Read, Grep, Glob
```

# Three generations, and they fail differently

Grep-class rules match syntax and cannot see data flow, so they flag the
parameterised query and the constant insert. Taint analysis follows source to
sink and is precise on the bugs it models. A model reads intent and finds the
class neither of the others can express — an authorization defect, where nothing
is malformed and the code is simply wrong about who may do what.

## When to use this

Choosing or defending a static analysis stack, and any time somebody proposes
replacing one generation with another rather than layering them.

## Procedure

**1 — Assemble a corpus with known ground truth.** Real bugs, safe lookalikes
of each bug, and at least one defect with no syntactic signature. The
lookalikes are what produce the precision number; without them every tool looks
perfect.

**2 — Run generation 1: pattern rules.** Record every hit and mark it against
ground truth. Precision here is usually about half, and the false positives are
the safe versions of the true positives.

**3 — Run generation 2: taint rules.** Source, sink, sanitiser. Expect high
precision and recall inside the model it has, and expect it to find nothing in
the file whose defect is not a flow.

**4 — Run generation 3: a model, with confidence.** Give it the same code. Record
what it finds, its confidence, and — separately — anything it asserts that is
not in the file. That last column is the cost of this generation.

**5 — Report per generation and per defect class.** The useful output is not a
winner; it is which class each generation can and cannot express, and the
precision each pays for its recall.

## Output contract

```json
{
  "corpus": [{"file": "str", "defect": "str|null", "cwe": "str|null"}],
  "generations": [{"name": "grep|taint|model", "findings": 0, "true_positives": 0,
                   "precision": 0.0, "recall": 0.0, "hallucinated": 0}],
  "only_found_by": [{"defect": "str", "generation": "str"}],
  "recommendation": {"layers": ["str"], "why": "str"}
}
```

## Failure modes

- **A corpus with no safe lookalikes.** Precision becomes meaningless.
- **Comparing on recall alone.** Grep has excellent recall and unusable
  precision.
- **Not counting the model's hallucinations.** They are the reason generation 3
  needs generation 4 — verification.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/sast-generation-comparison/scripts/sast_generation_comparison.py
SCRIPT = "skills/appsec/sast-generation-comparison/scripts/sast_generation_comparison.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Grep produces 6 findings at 50% precision, flagging the parameterised query, the constant insert and the safe subprocess call. Taint rules find exactly the 3 real injection bugs at 100% precision and recall and find nothing in `authz.py`. The model finds the authorization bug at 0.82 confidence and hallucinates one SQL injection at 0.41. The audit agent then runs the rules everywhere and spends the model pass on one file of four — the one where history says risk lives and the rules were silent — emitting 4 findings with zero false positives, every model finding marked as a hypothesis. The last cell shows what the allocation costs when it loses: give `authz.py` no history and the authorization bug is never reviewed.

## Your turn

Two things, and the second is the one people skip. Point the stand-in at a real GLM-4.6 or Kimi K2 through Ollama and run it on `authz.py` ten times — the variance in what it reports, and in its confidence, decides whether you can gate on confidence at all. Then run Semgrep against one of your own repositories at your current ruleset and at seven packs, and count the difference. Whatever that number is, it has been the number all year.

---

**Next → [B2.4 · Deduplication and contextual verification](https://spbreed.github.io/cyber-commons/lessons/B2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*